In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

In [0]:
df.limit(10).display()

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:

df = df.withColumn(
    "CID",
    F.regexp_replace(col("CID"), "^NAS", "")
)

In [0]:
df.limit(10).display()

In [0]:

df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

In [0]:
df = df.withColumn("BDATE",
              F.when(col("BDATE") > F.current_date(), None)
              .otherwise(col("BDATE")))

In [0]:

RENAME_MAP = {
    "cid": "customer_key",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.silver.erp_customers")